In [ ]:
import numpy as np
import pandas as pd
import os
from pathlib import Path
from tqdm import tqdm

import lightgbm as lgb
from lightgbm import LGBMRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.metrics import make_scorer
from sklearn.multioutput import MultiOutputRegressor

import torch
from torch.utils.data import DataLoader
import torchvision.transforms as T
from PIL import Image
from transformers import AutoModel
import torchvision.models as models


In [ ]:
TARGETS = ['Dry_Green_g', 'Dry_Dead_g', 'Dry_Clover_g', 'GDM_g', 'Dry_Total_g']
WEIGHTS = np.array([0.1, 0.1, 0.1, 0.2, 0.5])

def weighted_r2_metric(y_true, y_pred):
    ss_res = np.sum(WEIGHTS * np.sum((y_true - y_pred)**2, axis=0))
    global_mean = np.average(np.mean(y_true, axis=0), weights=WEIGHTS)
    ss_tot = np.sum(WEIGHTS * np.sum((y_true - global_mean)**2, axis=0))
    return 1 - (ss_res / ss_tot)

weighted_scorer = make_scorer(weighted_r2_metric, greater_is_better=True)

In [ ]:
@torch.no_grad()
def extract_efficientnet_features(df, model, device, base_path):
    """EfficientNetでTTA特徴抽出 -> DataFrame"""
    tta_transforms = [
        T.Compose([T.Resize(256), T.CenterCrop(224), T.ToTensor(), T.Normalize(MEAN, STD)]),
        T.Compose([T.Resize(256), T.CenterCrop(224), T.RandomHorizontalFlip(p=1.0), T.ToTensor(), T.Normalize(MEAN, STD)])
    ]
    unique_paths = df['image_path'].unique()
    all_feats = []
    for path in tqdm(unique_paths, desc="EfficientNet Feature Extraction"):
        img = Image.open(base_path / path).convert("RGB")
        tta_results = []
        for aug in tta_transforms:
            img_tensor = aug(img).unsqueeze(0).to(device)
            # EfficientNetの特徴抽出（分類層の前まで）
            features = model.features(img_tensor)
            # Global Average Pooling
            features = torch.nn.functional.adaptive_avg_pool2d(features, (1, 1))
            features = features.view(features.size(0), -1).cpu().numpy()
            tta_results.append(features)
        all_feats.append(np.mean(tta_results, axis=0))
    feat_matrix = np.vstack(all_feats)
    feat_cols = [f"effnet_feat_{i}" for i in range(feat_matrix.shape[1])]
    return pd.DataFrame(feat_matrix, columns=feat_cols), unique_paths


In [ ]:
@torch.no_grad()
def extract_combined_features(df, dinov2_model, effnet_model, device, base_path, use_effnet=True):
    """DINOv2とEfficientNetの両方から特徴量を抽出して統合"""
    # DINOv2特徴量を抽出
    dinov2_df, paths = extract_dense_features(df, dinov2_model, device, base_path)
    
    if use_effnet and effnet_model is not None:
        # EfficientNet特徴量を抽出
        effnet_df, _ = extract_efficientnet_features(df, effnet_model, device, base_path)
        # 特徴量を連結
        combined_df = pd.concat([dinov2_df, effnet_df], axis=1)
    else:
        # EfficientNetが使えない場合はDINOv2のみ
        combined_df = dinov2_df
    
    return combined_df, paths


In [13]:
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

@torch.no_grad()
def extract_dense_features(df, model, device, base_path):
    """TTA 提取 dense 特征 -> DataFrame"""
    tta_transforms = [
        T.Compose([T.Resize(256), T.CenterCrop(224), T.ToTensor(), T.Normalize(MEAN, STD)]),
        T.Compose([T.Resize(256), T.CenterCrop(224), T.RandomHorizontalFlip(p=1.0), T.ToTensor(), T.Normalize(MEAN, STD)])
    ]
    unique_paths = df['image_path'].unique()
    all_feats = []
    for path in tqdm(unique_paths, desc="GPU Feature Extraction"):
        img = Image.open(base_path / path).convert("RGB")
        tta_results = [
            model(aug(img).unsqueeze(0).to(device)).last_hidden_state[:, 1:, :].mean(dim=1).cpu().numpy()
            for aug in tta_transforms
        ]
        all_feats.append(np.mean(tta_results, axis=0))
    feat_matrix = np.vstack(all_feats)
    feat_cols = [f"feat_{i}" for i in range(feat_matrix.shape[1])]
    return pd.DataFrame(feat_matrix, columns=feat_cols), unique_paths

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# パス設定（ローカル環境とKaggle環境の両方に対応）
# Kaggle環境かどうかを判定
is_kaggle = os.path.exists("/kaggle/input")

if is_kaggle:
    base_path = Path("/kaggle/input/csiro-biomass")
    model_dir = os.path.abspath("/kaggle/input/dinov2/pytorch/large/1")
    notebook_dir = Path("/kaggle/working")
else:
    # ローカル環境
    # ノートブックのディレクトリを取得（CSIROBiomassディレクトリを想定）
    notebook_dir = Path.cwd()
    if notebook_dir.name != "CSIROBiomass":
        # 親ディレクトリを確認
        if (notebook_dir / "CSIROBiomass").exists():
            notebook_dir = notebook_dir / "CSIROBiomass"
        elif (notebook_dir.parent / "CSIROBiomass").exists():
            notebook_dir = notebook_dir.parent / "CSIROBiomass"
    
    # ローカル環境ではデータパスを設定（必要に応じて変更）
    base_path = notebook_dir / "data"  # ローカル環境のデータパス
    model_dir = notebook_dir / "models" / "dinov2"  # DINOv2モデルのパス（ローカル環境用）

# DINOv2モデルの読み込み
# ローカル環境ではDINOv2モデルが存在しない場合があるため、エラーハンドリングを追加
try:
    dinov2_model = AutoModel.from_pretrained(
        model_dir, local_files_only=True, trust_remote_code=True
    ).to(device).eval()
    print(f"✓ DINOv2 model loaded from: {model_dir}")
except Exception as e:
    if is_kaggle:
        raise e  # Kaggle環境では必須
    else:
        print(f"⚠ DINOv2 model not found at {model_dir}")
        print("  For local execution, you may need to download DINOv2 model separately.")
        # ローカル環境ではDINOv2なしで実行できないため、エラーを発生
        raise FileNotFoundError(f"DINOv2 model not found. Please download it to {model_dir}")

# EfficientNet-B0モデルの読み込み（複数のパスを順にチェック）
from torchvision.models import EfficientNet_B0_Weights
from urllib.error import URLError
import warnings

# notebook_dirは既に上で設定済み

effnet_model = None
loaded = False

# 方法1: ローカルファイルから読み込む（プロジェクトのmodelsディレクトリ）
local_model_path = notebook_dir / "models" / "efficientnet_b0_rwightman-7f5810bc.pth"
if local_model_path.exists():
    try:
        effnet_model = models.efficientnet_b0(weights=None)
        effnet_model.load_state_dict(torch.load(local_model_path, map_location=device))
        effnet_model = effnet_model.to(device).eval()
        print(f"✓ EfficientNet-B0 loaded from local file: {local_model_path}")
        loaded = True
    except Exception as e:
        print(f"Failed to load from local file: {e}")

# 方法2: Kaggleの入力データから読み込む
if not loaded:
    kaggle_model_paths = [
        "/kaggle/input/efficientnet-b0/efficientnet_b0_rwightman-7f5810bc.pth",
        "/kaggle/input/pytorch-pretrained-models/efficientnet_b0_rwightman-7f5810bc.pth",
    ]
    for path in kaggle_model_paths:
        if os.path.exists(path):
            try:
                effnet_model = models.efficientnet_b0(weights=None)
                effnet_model.load_state_dict(torch.load(path, map_location=device))
                effnet_model = effnet_model.to(device).eval()
                print(f"✓ EfficientNet-B0 loaded from Kaggle input: {path}")
                loaded = True
                break
            except Exception as e:
                print(f"Failed to load from {path}: {e}")
                continue

# 方法3: キャッシュから読み込む
if not loaded:
    cache_path = Path.home() / ".cache" / "torch" / "hub" / "checkpoints" / "efficientnet_b0_rwightman-7f5810bc.pth"
    if cache_path.exists():
        try:
            effnet_model = models.efficientnet_b0(weights=None)
            effnet_model.load_state_dict(torch.load(cache_path, map_location=device))
            effnet_model = effnet_model.to(device).eval()
            print(f"✓ EfficientNet-B0 loaded from cache: {cache_path}")
            loaded = True
        except Exception as e:
            print(f"Failed to load from cache: {e}")

# 方法4: オンラインからダウンロード
if not loaded:
    try:
        effnet_model = models.efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1).to(device).eval()
        print("✓ EfficientNet-B0 downloaded and loaded")
        loaded = True
    except (URLError, OSError) as e:
        warnings.warn(f"Failed to download EfficientNet weights: {e}")

# 方法5: 重みなしモデル（フォールバック）
if not loaded:
    effnet_model = models.efficientnet_b0(weights=None).to(device).eval()
    print("⚠ EfficientNet-B0 loaded without pretrained weights (accuracy may be lower)")

train_df = pd.read_csv(base_path / "train.csv")
train_p = train_df.pivot_table(
    index="image_path", columns="target_name", values="target"
).reset_index()

# DINOv2とEfficientNetの両方から特徴量を抽出して統合
X_train_df, _ = extract_combined_features(train_p, dinov2_model, effnet_model, device, base_path)
Y_train_log = np.log1p(train_p[TARGETS].values)

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train_df), columns=X_train_df.columns
)

# 特徴量の次元削減（PCA）で過学習を抑制
from sklearn.decomposition import PCA

# PCAの次元数を訓練サンプル数以下に動的に設定
# 説明分散比95%を達成する最小次元数を自動選択
n_samples = X_train_scaled.shape[0]
n_features = X_train_scaled.shape[1]

# 説明分散比の目標（95%）で次元数を決定
pca_for_selection = PCA(random_state=42)
pca_for_selection.fit(X_train_scaled)

cumsum_var = np.cumsum(pca_for_selection.explained_variance_ratio_)
target_variance = 0.95  # 95%の分散を説明
n_components = np.argmax(cumsum_var >= target_variance) + 1
# ただし、訓練サンプル数と256のうち小さい方に制限（過学習防止）
n_components = min(n_components, n_samples - 1, 256)  # 最大256次元に制限
n_components = max(n_components, 50)  # 最小50次元は保持

pca = PCA(n_components=n_components, random_state=42)
X_train_pca = pca.fit_transform(X_train_scaled)
X_train_final = pd.DataFrame(
    X_train_pca,
    columns=[f"pca_{i}" for i in range(n_components)]
)

# 説明分散比を確認
explained_variance = pca.explained_variance_ratio_.sum()
print(f"\nPCA: {X_train_scaled.shape[1]} dimensions -> {X_train_final.shape[1]} dimensions")
print(f"Training samples: {n_samples}, Selected components: {n_components}")
print(f"Explained variance ratio: {explained_variance:.4f} ({explained_variance*100:.2f}%)")

HFValidationError: Repo id must be in the form 'repo_name' or 'namespace/repo_name': '/kaggle/input/dinov2/pytorch/large/1'. Use `repo_type` argument if needed.

In [ ]:
# 正則化を強化したハイパーパラメータ（過学習対策を強化）
# GridSearchCVではearly_stopping_roundsが使用できないため、n_estimatorsを適切に設定
# パラメータグリッドを削減してエラーを回避（必要に応じて後で拡張可能）
param_grid = {
    "estimator__learning_rate": [0.01, 0.03],  # 学習率を下げる
    "estimator__num_leaves": [31],  # 1つの値に削減
    "estimator__n_estimators": [100, 150],  # 2つの値に削減
    "estimator__max_depth": [5, 7],  # -1を削除（深さを制限）
    "estimator__min_child_samples": [30, 50],  # 葉の最小サンプル数を増やす
    "estimator__subsample": [0.7, 0.8],  # サブサンプリングを強化
    "estimator__colsample_bytree": [0.7, 0.8],  # 特徴量のサブサンプリングを強化
    "estimator__reg_alpha": [0.5, 1.0],  # L1正則化を強化
    "estimator__reg_lambda": [0.5, 1.0],  # L2正則化を強化
    "estimator__random_state": [42]
}
base_lgbm = MultiOutputRegressor(
    LGBMRegressor(
        verbosity=-1
    )
)
grid_search = GridSearchCV(
    base_lgbm,
    param_grid,
    cv=KFold(n_splits=5, shuffle=True, random_state=42),
    scoring=weighted_scorer,
    n_jobs=-1,
    verbose=1,  # 進捗を表示
    error_score='raise'  # エラーの詳細を確認
)
# PCA適用後の特徴量を使用
grid_search.fit(X_train_final, Y_train_log)

print("\n" + "="*50)
print(f"Best LGBM Params: {grid_search.best_params_}")
print(f"Best CV Weighted R2: {grid_search.best_score_:.4f}")
print("="*50)

In [ ]:
test_df = pd.read_csv(base_path / "test.csv")
test_uniq = pd.DataFrame({"image_path": test_df["image_path"].unique()})
# DINOv2とEfficientNetの両方から特徴量を抽出して統合
X_test_df, test_ids = extract_combined_features(test_uniq, dinov2_model, effnet_model, device, base_path)
X_test_scaled = pd.DataFrame(scaler.transform(X_test_df), columns=X_test_df.columns)

# テストデータにも同じPCAを適用
X_test_pca = pca.transform(X_test_scaled)
X_test_final = pd.DataFrame(
    X_test_pca,
    columns=[f"pca_{i}" for i in range(n_components)]
)

test_preds_log = grid_search.best_estimator_.predict(X_test_final)
avg_preds = np.maximum(np.expm1(test_preds_log), 0)

In [ ]:
final_rows = []
for i, path in enumerate(test_ids):
    image_id = Path(path).stem
    for j, target_name in enumerate(TARGETS):
        final_rows.append(
            {"sample_id": f"{image_id}__{target_name}", "target": avg_preds[i, j]}
        )
pd.DataFrame(final_rows).to_csv("submission.csv", index=False)
print("\n LGBM submission saved to submission.csv")